# Milestone 3 — The NOVA Brain: LLM + Planner + Tool Router
### Stage 5 (part 1) of NOVA: turning transcribed text into a reasoned response

This notebook teaches the concepts behind `nova/brain/llm_client.py`, `planner.py`, and `router.py` before you rely on `NovaBrain` in the real pipeline. We'll cover:

1. What a local LLM server (Ollama) actually is, and why NOVA runs one locally instead of calling a cloud API
2. Prompt engineering for structured output -- getting an LLM to reliably return JSON, not prose
3. "Custom" agent framework vs LangChain -- what you're actually trading off
4. Building and testing the Planner's decision logic
5. The Tool Router: how a JSON `{"tool_name": ...}` becomes a real Python function call
6. The full loop, traced end to end

> **Note on this environment:** no Ollama server is running in this sandbox, and there's no internet to reach one. Every cell either runs fully self-contained using a fake LLM function that mimics Ollama's output, or is clearly marked "run this on your own machine" once you have `ollama serve` running locally.


In [1]:
# Install locally (not in this sandbox):
# pip install ollama
# and separately: install Ollama itself from https://ollama.com, then:
#   ollama pull llama3.1
#   ollama serve   (often already running as a background service after install)

import json
print("Ready.")


Ready.


## 1. What is Ollama, and why local?

Ollama runs an LLM (here, Meta's Llama 3.1) as a small local HTTP server on your machine (`http://localhost:11434` by default). Your Python code sends it a list of chat messages; it sends back a generated reply -- the same request/response shape as OpenAI's API, just pointed at your own machine instead of the cloud.

Why this matters for NOVA specifically:
- **Privacy**: nothing you say to NOVA leaves your laptop
- **No API costs**: every request is free once the model is downloaded
- **Offline capable**: works without internet (aside from the one-time model download)
- **Latency**: no network round-trip -- just local inference time

The tradeoff: a locally-runnable model (llama3.1, 8B parameters) is less capable than the largest cloud models. For NOVA's use case -- short commands, tool selection, not open-ended essay writing -- this tradeoff is usually fine.


In [2]:
# --- RUN THIS ON YOUR OWN MACHINE, with `ollama serve` running ---
import ollama

client = ollama.Client(host="http://localhost:11434")

response = client.chat(
    model="llama3.1",
    messages=[{"role": "user", "content": "Say hello in exactly 3 words."}],
    options={"temperature": 0.3},
)
print(response["message"]["content"])


ResponseError: model 'llama3.1' not found (status code: 404)

## 2. Prompt engineering for structured output

Left to its own devices, an LLM will happily reply with prose: *"Sure! Right now it's 3:15 PM."* That's great for a human but useless for code that needs to programmatically decide "should I call a tool?"

The fix: **tell it exactly what shape to respond in**, and give it no other option. This is the entire idea behind your `Planner`'s system prompt -- it doesn't ask the LLM to *decide and explain*, it asks the LLM to output *one JSON object, nothing else*.


In [ ]:
SYSTEM_PROMPT = """You are NOVA's planning module. Given the user's request, decide whether to \
answer directly or call a tool.

Respond with ONLY a single JSON object, no other text, in one of these two shapes:
{"action": "answer", "answer": "<your direct reply to the user>"}
{"action": "tool", "tool_name": "<tool name>", "tool_args": {"<arg>": "<value>"}}

Available tools:
- get_current_time: Returns the current local time. No arguments needed.

If no tool is relevant, use the "answer" shape."""

print(SYSTEM_PROMPT)


### Why this is fragile, and how `Planner` defends against it

Even with a strict instruction, LLMs occasionally ignore it -- adding a stray sentence before the JSON, using single quotes, etc. Your real `Planner._parse()` handles this with a simple, honest fallback: try `json.loads()`, and if it fails, treat the *entire raw response* as a direct answer instead of crashing. A worse answer beats a crashed assistant.


In [3]:
def parse_planner_output(raw: str) -> dict:
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        print(f"WARNING: not valid JSON, falling back to direct answer: {raw!r}")
        return {"action": "answer", "answer": raw.strip()}

    action = data.get("action", "answer")
    if action == "tool" and data.get("tool_name"):
        return {"action": "tool", "tool_name": data["tool_name"], "tool_args": data.get("tool_args") or {}}
    return {"action": "answer", "answer": data.get("answer", "")}


# Well-formed cases
print(parse_planner_output('{"action": "answer", "answer": "Hello there!"}'))
print(parse_planner_output('{"action": "tool", "tool_name": "get_current_time", "tool_args": {}}'))

# The LLM ignored instructions and just chatted -- still handled gracefully
print(parse_planner_output("Sure, here's the time: it's 3pm!"))


{'action': 'answer', 'answer': 'Hello there!'}
{'action': 'tool', 'tool_name': 'get_current_time', 'tool_args': {}}
{'action': 'answer', 'answer': "Sure, here's the time: it's 3pm!"}


## 3. "Custom" agent framework vs LangChain

The diagram lists "LangChain / Custom" as the Agent Framework option. NOVA's `Planner` is the **Custom** path: a single LLM call with a JSON contract, parsed by hand.

| | Custom (what NOVA uses) | LangChain |
|---|---|---|
| **Transparency** | Every step is visible Python you wrote | Agent loop logic lives inside the framework |
| **Debuggability** | Print the raw LLM response, see exactly what happened | Requires understanding LangChain's internal abstractions to debug |
| **Dependencies** | Just the `ollama` client | A large framework with its own versioning/breaking changes |
| **Flexibility for learning** | You see every prompt, every parse step | Framework often hides prompt construction |
| **Multi-step reasoning (ReAct loops, chained tool calls)** | You'd have to build this yourself | Comes built-in |

For a project you're building to learn from, Custom is the better choice -- you can see and modify every piece. If NOVA later needs more complex multi-step tool chains (call tool A, look at result, decide to call tool B), that's when reaching for LangChain's agent executor starts to pay off.


## 4. The Tool Router: JSON to Python function call

Once the Planner decides `{"action": "tool", "tool_name": "get_current_time", "tool_args": {}}`, something has to turn that into an actual function call. That's the `ToolRouter` -- a dictionary mapping tool names to Python functions, plus a safe `execute()` that never lets a broken tool crash the whole assistant.


In [4]:
class Tool:
    def __init__(self, name, description, func):
        self.name, self.description, self.func = name, description, func

class ToolRouter:
    def __init__(self):
        self.tools = {}

    def register(self, tool):
        self.tools[tool.name] = tool
        print(f"Registered tool '{tool.name}'")

    def execute(self, tool_name, tool_args):
        tool = self.tools.get(tool_name)
        if tool is None:
            return f"(error: no such tool '{tool_name}')"
        try:
            return tool.func(**tool_args)
        except Exception as exc:
            return f"(error running tool '{tool_name}': {exc})"


import datetime
def get_current_time():
    return datetime.datetime.now().strftime("%I:%M %p")

router = ToolRouter()
router.register(Tool("get_current_time", "Returns the current local time.", get_current_time))

# Simulate what happens after the Planner decides to call this tool:
decision = parse_planner_output('{"action": "tool", "tool_name": "get_current_time", "tool_args": {}}')
result = router.execute(decision["tool_name"], decision["tool_args"])
print("Tool result:", result)


Registered tool 'get_current_time'
Tool result: 01:46 PM


In [5]:
# What happens with a tool that doesn't exist, or one that throws?
print(router.execute("nonexistent_tool", {}))

def broken_tool():
    raise ValueError("something went wrong inside the tool")

router.register(Tool("broken", "Always fails, for demonstration.", broken_tool))
print(router.execute("broken", {}))


(error: no such tool 'nonexistent_tool')
Registered tool 'broken'
(error running tool 'broken': something went wrong inside the tool)


Notice neither error crashed anything -- both returned a plain error *string*. This is deliberate: a single misbehaving tool should degrade NOVA's answer quality for that one turn, not take down the whole assistant.


## 5. The full loop, traced end to end

```
user_text = "what time is it?"
        │
        ▼
Planner.decide(user_text)
   ├─ builds system prompt listing all registered tools
   ├─ LLMClient.chat([system_prompt, ...context, user_text])
   ├─ LLM replies: {"action": "tool", "tool_name": "get_current_time", "tool_args": {}}
   └─ parsed into PlanDecision(action="tool", tool_name="get_current_time", tool_args={})
        │
        ▼
ToolRouter.execute("get_current_time", {})  -> "3:15 PM"
        │
        ▼
LLMClient.chat([...original context, "tool result: 3:15 PM", "now answer naturally"])
        │
        ▼
"It's currently 3:15 PM."   <- final response, returned to the user
```

Two LLM calls happen for a tool-using turn: one to *decide*, one to *phrase the final answer naturally* using the tool's result. A direct-answer turn (no tool needed) only costs one LLM call.


In [6]:
# A minimal end-to-end simulation using a scripted fake LLM (no Ollama needed to run this cell)

class FakeLLM:
    def __init__(self, responses):
        self._responses = list(responses)
    def chat(self, messages):
        return self._responses.pop(0) if self._responses else "(no more canned responses)"

fake_llm = FakeLLM([
    '{"action": "tool", "tool_name": "get_current_time", "tool_args": {}}',
    "It's currently 3:15 PM.",
])

def think(user_text, router, llm):
    decision = parse_planner_output(llm.chat([{"role": "user", "content": user_text}]))
    if decision["action"] == "tool":
        result = router.execute(decision["tool_name"], decision["tool_args"])
        return llm.chat([{"role": "user", "content": f"tool result: {result}, now answer naturally"}])
    return decision["answer"]

print(think("what time is it?", router, fake_llm))


It's currently 3:15 PM.


### Exercises
1. Register a second tool (e.g. `def flip_a_coin(): import random; return random.choice(["heads", "tails"])`) and manually write the JSON decision an LLM *should* produce to call it -- then run it through `router.execute()`.
2. Once you have Ollama running locally, send the real system prompt from Section 2 with a user message like "what's 47 times 6?" -- does llama3.1 correctly choose "answer" instead of trying to invoke a nonexistent calculator tool? Try adding a `calculate` tool and see if it picks it up.
3. Modify `parse_planner_output` to also handle the case where `tool_args` contains a JSON string instead of a dict (a common LLM quirk) -- add a `json.loads()` fallback for that field specifically.

**Next notebook:** Milestone 4 — Memory & RAG (Short-term Context, Long-term Memory, and semantic search with ChromaDB).
